# Λόγια ή πράξεις; Πώς ταξιδεύει η βλάβη σε ένα δίκτυο LLM

Το σχέδιο και οι κανόνες ανάγνωσης είναι στο `docs/contagion_plan.md`,
γραμμένα πριν από τα runs.

- **Δίκτυο:** γραμμή 6 πρακτόρων, 0–1–2–3–4–5. Ο πράκτορας 0 είναι η «πηγή»,
  και ο πράκτορας d απέχει d βήματα από αυτήν.
- **Η πηγή** δεν καλεί μοντέλο: παίζει σταθερή κίνηση και στέλνει μηνύματα
  από τη λίστα του ίδιου μοντέλου στο corpus (`pools/`).

| κελί | λόγια της πηγής | πράξεις της πηγής |
|---|---|---|
| `source_cc` | συνεργατικά | πάντα συνεργασία (έλεγχος) |
| `source_hc` | εχθρικά | πάντα συνεργασία («φωνακλάς») |
| `source_cd` | συνεργατικά | πάντα αποστασία («ψεύτης») |
| `source_hd` | εχθρικά | πάντα αποστασία |
| `baseline` | ο 0 είναι κανονικός πράκτορας LLM | |

- **Ρυθμίσεις:** PD, 16 γύροι, 5 runs ανά κελί, διορθωμένο prompt δρομολόγησης.
- **Μοντέλα:** Qwen2.5-7B στην κάρτα 0 και Llama-3.1-8B στην κάρτα 1, παράλληλα.

## Setup

1. Settings → Accelerator → **GPU T4 x2** (δύο κάρτες, ένα μοντέλο στην καθεμία)
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach** (το Llama είναι gated)

Μετά **Save Version → Save & Run All**. Τα δύο μοντέλα τρέχουν ταυτόχρονα, και
στις 11,5 ώρες το notebook σταματά μόνο του και πακετάρει ό,τι έχει τελειώσει.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import os, time, torch
assert torch.cuda.is_available(), 'GPU not enabled -- Settings -> Accelerator -> GPU T4 x2'
N_GPU = torch.cuda.device_count()
assert N_GPU >= 2, f'Χρειάζονται δύο κάρτες, βλέπω {N_GPU}: Accelerator -> GPU T4 x2'
print('GPU:', N_GPU, 'x', torch.cuda.get_device_name(0))
T_START = time.time()
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    raise SystemExit(f'Λείπει το HF_TOKEN (χρειάζεται για το Llama): {e}')

In [ ]:
import subprocess
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# Gate: χωρίς τον κώδικα της πηγής θα έτρεχαν απλές γραμμές.
assert 'source_cc' in open('run_all_scenarios.py', encoding='utf-8').read(), \
    'Το repo δεν έχει τα κελιά της πηγής -- κάνε git push πρώτα.'
MODELS = {0: 'Qwen/Qwen2.5-7B-Instruct', 1: 'meta-llama/Llama-3.1-8B-Instruct'}
for m in MODELS.values():
    for kind in ('hostile', 'cooperative'):
        p = f"pools/{m.split('/')[-1]}_{kind}.json"
        assert os.path.exists(p), f'Λείπει η λίστα {p}'
print('gate ok')

## Τα δύο μοντέλα, παράλληλα

Το καθένα τρέχει όλα τα 5 κελιά × 5 runs = 25 runs. Το δεύτερο ξεκινά αφού το
πρώτο έχει φορτώσει στην κάρτα του. Κάθε 10 λεπτά τυπώνεται η πρόοδος.

In [ ]:
import sys
RESULTS = '/kaggle/working/results'
LOGS = '/kaggle/working/logs'
os.makedirs(LOGS, exist_ok=True)
DEADLINE = T_START + 11.5 * 3600

def cmd(model):
    return [sys.executable, 'campaign.py', '--model', model, '--session', 'SRC',
            '--topology', 'line', '--n-agents', '6', '--topology-aware-comm-prompt',
            '--games', 'pd', '--conditions', 'cheap_talk', '--n-runs', '5', '--n-rounds', '16',
            '--source-pools', 'pools', '--out-root', RESULTS, '--zip-mirror', '']

procs, logs = {}, {}
for gpu, model in MODELS.items():
    log = f"{LOGS}/{model.split('/')[-1]}.log"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               TOKENIZERS_PARALLELISM='false')
    procs[gpu] = subprocess.Popen(cmd(model), stdout=open(log, 'w', encoding='utf-8'),
                                  stderr=subprocess.STDOUT, env=env)
    logs[gpu] = log
    print(f'κάρτα {gpu}: {model} -> {log}')
    while procs[gpu].poll() is None and 'Model loaded' not in open(log, encoding='utf-8').read():
        time.sleep(10)

def last(log):
    lines = [l for l in open(log, encoding='utf-8').read().splitlines()
             if 'saved ->' in l or 'SCENARIO' in l or 'FAIL' in l]
    return lines[-1].strip()[:110] if lines else '(φόρτωση)'

while any(p.poll() is None for p in procs.values()):
    if time.time() > DEADLINE:
        for p in procs.values():
            p.terminate()
        print('11,5 ώρες: σταμάτησα. Ό,τι τελείωσε πακετάρεται παρακάτω.')
        break
    time.sleep(600)
    print(time.strftime('%H:%M'), ' | '.join(f'κάρτα {g}: {last(l)}' for g, l in logs.items()), flush=True)

for gpu, log in logs.items():
    print(f'--- κάρτα {gpu}, τέλος log ---')
    print(open(log, encoding='utf-8').read()[-1500:])

## Ανάλυση με τους κανόνες του σχεδίου, και zip

Το `contagion_report.py` εφαρμόζει τους κανόνες R1–R3 όπως είναι γραμμένοι στο
σχέδιο. Οι ετυμηγορίες βγαίνουν μόνο όταν έχουν τρέξει και τα δύο μοντέλα.

In [ ]:
import glob, shutil
roots = sorted(glob.glob(f'{RESULTS}/*_line_n6_commfix_cheaptalk'))
print('trees:', roots)
r = subprocess.run([sys.executable, 'contagion_report.py', '--roots', *roots,
                    '--csv', '/kaggle/working/contagion_summary.csv'],
                   capture_output=True, text=True)
print(r.stdout[-12000:]); print(r.stderr[-2000:])
shutil.make_archive('/kaggle/working/contagion_results', 'zip', RESULTS)
print('κατέβασε: /kaggle/working/contagion_results.zip',
      f"({os.path.getsize('/kaggle/working/contagion_results.zip')/1e6:.1f} MB)")

## Μετά

Κατέβασε το `contagion_results.zip` και τα logs, και αποσυμπίεσέ το σε νέο φάκελο
`diplomatikh/contagion_runs/`. Τα runs **δεν** μπαίνουν στο tracker: είναι
παραλλαγές (σενάριο πηγής, έξι πράκτορες), όπως τα `harm_source/`, και το ingest
σωστά τα αρνείται. Η ανάλυση ξανατρέχει τοπικά (από τον φάκελο `diplomatikh`):

```
python cheaptalk_bench/contagion_report.py --roots contagion_runs/*_line_n6_commfix_cheaptalk
```